In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from transformer_lens import HookedTransformer
import torch

import json
import time
import os

from src.neuron_heads import head_attribution_over_all_data
from src.datahandlers import ActivatingDataset
from src.utils import tuple_str_to_tuple

In [3]:

filename = "2026-06-15_20-00-01_gpt2-small-random_test"
is_random = "-random" # otherwise, ""
train = False


with open(f'../experiment_data/3_trimmed_texts/{filename}.json') as f:
    text_metadata = json.load(f)

neurons = [tuple_str_to_tuple(x) for x in text_metadata['neuron_to_trunc_data'].keys()]

In [4]:
import pickle
dataset_text_test_path = "../experiment_data/text_dict_test.pkl"
dataset_text_train_path = "../experiment_data/text_dict_train.pkl"
if train:
    with open(dataset_text_train_path, 'rb') as f:
        dataset = pickle.load(f)

    print("Train loaded!")

else:
    with open(dataset_text_test_path, 'rb') as f:
        dataset = pickle.load(f)

    # removing samples that were also in the training dataset
    dataset.pop(9272) 
    dataset.pop(7138)
    print("Test loaded!")
    

data = ActivatingDataset(text_metadata['neuron_to_trunc_data'], dataset)
data.remove_prompts_longer_than(100)

Test loaded!


In [5]:
# Parameters
parameters = text_metadata['parameters']
model_name = parameters['model_name']

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [6]:
model = HookedTransformer.from_pretrained(
    model_name,
    center_unembed=True,
    center_writing_weights=True,
    fold_ln=True,
    # refactor_factored_attn_matrices=True,
    device=device,
)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loaded pretrained model gpt2-small into HookedTransformer


In [7]:
import gc
gc.collect()
torch.cuda.empty_cache()

In [8]:
with torch.no_grad():
    head_attribution_dict, neuron_prompt_head_scores = head_attribution_over_all_data(
        model=model,
        data=data,
        device=device,
        neurons=neurons,
        batch_size=16,
    )

  0%|          | 0/60 [00:00<?, ?it/s]

19


  2%|▏         | 1/60 [00:00<00:44,  1.33it/s]

17


  5%|▌         | 3/60 [00:01<00:29,  1.95it/s]

...

 97%|█████████▋| 58/60 [00:29<00:00,  2.26it/s]

19


 98%|█████████▊| 59/60 [00:29<00:00,  2.14it/s]

20


100%|██████████| 60/60 [00:30<00:00,  1.97it/s]


In [9]:
output = {
    'parameters': parameters,
    'head_attributions': head_attribution_dict,
    'prior_filename': filename,
}

train_str = "_train" if train else "_test"

timestamp = time.strftime("%Y-%m-%d_%H-%M-%S", time.localtime(int(time.time())))
new_filename = f"{timestamp}_{model_name}{is_random}{train_str}.json"

folder_path = "../experiment_data/4_head_attributions"
os.makedirs(folder_path, exist_ok=True)

with open(os.path.join(folder_path, new_filename) , 'w') as f:
    json.dump(output, f)

In [9]:
output = {
    'parameters': parameters,
    'neuron_prompt_head_scores': neuron_prompt_head_scores,
    'prior_filename' : new_filename
}

train_str = "train" if train else "test"

# Save json to ../experiment_data/2_max_activating_texts
timestamp = time.strftime("%Y-%m-%d_%H-%M-%S", time.localtime(int(time.time())))
new_filename = f"{timestamp}_{model_name}_{train_str}_dot_product.json"

folder_path = "../experiment_data/5_head_attribution_scores"
os.makedirs(folder_path, exist_ok=True)

with open(os.path.join(folder_path, new_filename) , 'w') as f:
    json.dump(output, f)